# Traditional ML for Daily Hotspot Prediction
## Train on 2023 → Test on 2024

**Goal:** Predict the **number of hotspots tomorrow** in the Chiang Mai study area.

In this lab, we deliberately use a **time-based split**:
- **Training set:** 2023
- **Test set:** 2024

This is more realistic than randomly mixing past and future observations.

> **Note:** The supplied hotspot datasets use an interim Chiang Mai bounding-box filter. They are suitable for teaching the ML workflow, but a province-polygon filter should be used for a final scientific study.


## Learning objectives
After this lab, students should be able to:
1. Define features (`X`) and a regression target (`y`).
2. Explain why time-series data should not be randomly split.
3. Build a simple persistence baseline.
4. Train **Linear Regression** and **Random Forest Regression**.
5. Evaluate models using **MAE, RMSE, and R²**.
6. Compare the baseline, linear model, and nonlinear model on future data.


In [ ]:
# 1. Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)


## 2. Load the datasets from GitHub

The 2023 and 2024 CSV files are stored in the course GitHub repository.

`pandas.read_csv()` can read the **raw GitHub files directly**, so you do not need to download or upload the CSV files manually.


In [ ]:
# 2. Load data directly from GitHub

base_url = "https://raw.githubusercontent.com/pvateekul/workshop/main/20261008_AirQuality/lab"

train_url = f"{base_url}/chiang_mai_hotspot_ml_2023_interim.csv"
test_url  = f"{base_url}/chiang_mai_hotspot_ml_2024_interim.csv"

train_df = pd.read_csv(train_url, parse_dates=["date"])
test_df  = pd.read_csv(test_url, parse_dates=["date"])

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head())


In [ ]:
# Check the time periods
print("Training period:", train_df["date"].min(), "to", train_df["date"].max())
print("Testing period :", test_df["date"].min(), "to", test_df["date"].max())


## 3. Explore the target

The regression target is:

`target_hotspot_count_next_day`

For a row representing day **t**, this value is the number of hotspots on day **t+1**.


In [ ]:
# Plot daily hotspot counts
plt.figure(figsize=(14, 4))
plt.plot(train_df["date"], train_df["hotspot_count"], label="2023")
plt.plot(test_df["date"], test_df["hotspot_count"], label="2024")
plt.xlabel("Date")
plt.ylabel("Hotspot count")
plt.title("Daily Hotspot Counts")
plt.legend()
plt.show()

print("2023 target statistics")
display(train_df["target_hotspot_count_next_day"].describe())

print("2024 target statistics")
display(test_df["target_hotspot_count_next_day"].describe())


## 4. Define inputs and target

We use information available on or before today to predict tomorrow.

### Inputs
- Today's hotspot count and fire intensity
- Day/night hotspot counts
- Historical hotspot lags
- Rolling historical hotspot averages

### Target
- Tomorrow's hotspot count

**Important:** We do **not** use `target_hotspot_next_day_binary` as an input because that would leak information about the future.


In [ ]:
features = [
    "hotspot_count",
    "frp_sum",
    "frp_mean",
    "high_confidence_count",
    "day_hotspot_count",
    "night_hotspot_count",
    "hotspot_lag_1d",
    "hotspot_lag_2d",
    "hotspot_lag_3d",
    "hotspot_lag_7d",
    "hotspot_lag_14d",
    "hotspot_rolling_mean_3d",
    "hotspot_rolling_mean_7d",
]

target = "target_hotspot_count_next_day"

X_train = train_df[features].copy()
y_train = train_df[target].copy()

X_test = test_df[features].copy()
y_test = test_df[target].copy()

# The last day of each annual file has no next-day target.
train_mask = y_train.notna()
test_mask = y_test.notna()

X_train = X_train.loc[train_mask]
y_train = y_train.loc[train_mask]

X_test = X_test.loc[test_mask]
y_test = y_test.loc[test_mask]
test_dates = test_df.loc[test_mask, "date"]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)


## 5. Why not `train_test_split()`?

For ordinary IID data, random splitting is common. For forecasting, however, random splitting can allow information from later dates to influence training.

Here we simulate a realistic scenario:

> **Learn from the past (2023), then predict the future (2024).**


## 6. Establish a baseline

Before using ML, always ask:

> **Can the ML model beat a very simple prediction rule?**

A **persistence baseline** predicts that tomorrow's hotspot count will be the same as today's hotspot count.


In [ ]:
baseline_pred = X_test["hotspot_count"].to_numpy()

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, baseline_pred)
pd.DataFrame([baseline_metrics], index=["Persistence baseline"])


## 7. Train two traditional ML models

To keep the lab simple, we compare only two ML approaches:

- **Linear Regression** — a simple linear model that is easy to interpret.
- **Random Forest Regression** — a nonlinear ensemble of decision trees.

Together with the **Persistence Baseline**, this gives us a clear comparison:

**Simple rule → Linear ML → Nonlinear ML**

Missing lag values at the beginning of the year are imputed using the median calculated from the **training data only**.


In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),
}

predictions = {}
results = []

# Add the simple persistence baseline
results.append({"Model": "Persistence baseline", **baseline_metrics})

# Train and evaluate each ML model
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    # Hotspot counts cannot be negative.
    pred = np.clip(pred, 0, None)

    predictions[name] = pred
    results.append({"Model": name, **regression_metrics(y_test, pred)})

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
results_df


## 8. Interpret the evaluation metrics

### MAE — Mean Absolute Error
Average absolute difference between prediction and actual count.

**Lower is better.**

### RMSE — Root Mean Squared Error
Similar to MAE but penalizes large errors more strongly.

**Lower is better.**

### R² — Coefficient of Determination
Measures how much variation in the target is explained by the model.

- `1.0` = perfect prediction
- `0.0` = similar to predicting the test-set mean
- `< 0` = worse than that mean-based reference

For highly variable hotspot counts, examine **all three metrics**, not R² alone.


In [ ]:
# Display model comparison
results_df.style.format({
    "MAE": "{:.2f}",
    "RMSE": "{:.2f}",
    "R2": "{:.3f}"
})


## 9. Plot Actual vs Predicted

A good model should follow both:
- low-hotspot periods, and
- major hotspot peaks.


In [ ]:
best_ml_name = results_df.loc[
    results_df["Model"] != "Persistence baseline", "Model"
].iloc[0]

best_pred = predictions[best_ml_name]

plt.figure(figsize=(15, 5))
plt.plot(test_dates, y_test.to_numpy(), label="Actual", linewidth=2)
plt.plot(test_dates, best_pred, label=f"Predicted: {best_ml_name}", alpha=0.8)
plt.xlabel("Date")
plt.ylabel("Tomorrow's hotspot count")
plt.title(f"2024 Actual vs Predicted — {best_ml_name}")
plt.legend()
plt.show()


In [ ]:
# Zoom into the main burning-season window
mask = (test_dates >= "2024-02-01") & (test_dates <= "2024-05-31")

plt.figure(figsize=(15, 5))
plt.plot(test_dates[mask], y_test.to_numpy()[mask], label="Actual", linewidth=2)
plt.plot(test_dates[mask], best_pred[mask], label="Predicted", alpha=0.8)
plt.xlabel("Date")
plt.ylabel("Tomorrow's hotspot count")
plt.title(f"Burning Season: Actual vs Predicted — {best_ml_name}")
plt.legend()
plt.show()


## 10. Random Forest feature importance

Feature importance can help answer:

> Which historical fire variables contribute most to the model?

Remember: **feature importance does not imply causation**.


In [ ]:
rf = models["Random Forest"].named_steps["model"]

importance_df = (
    pd.DataFrame({
        "Feature": features,
        "Importance": rf.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

display(importance_df)

plt.figure(figsize=(9, 6))
plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
plt.xlabel("Feature importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()


## 11. Error analysis

Looking only at one overall metric can hide important behavior. Let's inspect the days with the largest prediction errors.


In [ ]:
error_df = pd.DataFrame({
    "date": test_dates.to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": best_pred
})

error_df["absolute_error"] = np.abs(
    error_df["actual"] - error_df["predicted"]
)

error_df.sort_values("absolute_error", ascending=False).head(15)


## 12. Discussion questions

1. Which performs better: **Linear Regression** or **Random Forest**?
2. Do both ML models beat the **Persistence Baseline**?
3. Why might Random Forest perform better than Linear Regression?
4. During which periods are prediction errors largest?
5. Can either model accurately predict sudden hotspot peaks?
6. Why should we avoid randomly mixing 2023 and 2024?
7. What important variables are missing from this dataset?

### Next step: add weather
A stronger model could include:
- Temperature
- Relative humidity
- Rainfall
- Wind speed

Then we can investigate whether **weather information improves next-day hotspot prediction** beyond hotspot history alone.


## Take-home message

A forecasting model should be evaluated on **future data that was not used for training**.

In this lab:

**2023 → Train → Model → Predict 2024 → Compare with actual 2024**

Always compare ML against a simple baseline before concluding that the ML model is useful.
